# 05 — IVC Demo: Proving 1000 Fibonacci Steps

**Incrementally Verifiable Computation (IVC)** proves that a computation $f^n(x_0)$ was executed correctly. The prover maintains a running accumulator that folds in each new step.

$$\text{Step } i: z_{i+1} = f(z_i), \quad \text{acc}_{i+1} = \text{fold}(\text{acc}_i, \text{fresh}_i)$$

At the end, the verifier checks one accumulated Relaxed R1CS instance. The cost doesn't grow with $n$.

**Nova paper reference:** Section 5 (IVC from Folding Schemes)

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from nano_nova.field import GF
from nano_nova.examples import fibonacci_step_circuit, fibonacci_step_fn, fibonacci_witness_fn
from nano_nova.ivc import ivc_prove, ivc_verify

shape = fibonacci_step_circuit()

## Small Demo: 10 Steps

In [ ]:
z0 = GF(np.array([1, 1]))

t0 = time.time()
proof = ivc_prove(shape, fibonacci_step_fn, fibonacci_witness_fn, z0, num_steps=10)
prove_time = time.time() - t0

t0 = time.time()
valid = ivc_verify(shape, proof)
verify_time = time.time() - t0

print(f"IVC proof for 10 Fibonacci steps:")
print(f"  Final state: z_10 = ({int(proof.z_current[0])}, {int(proof.z_current[1])})")
print(f"  Valid: {valid}")
print(f"  Prove time:  {prove_time*1000:.1f} ms")
print(f"  Verify time: {verify_time*1000:.3f} ms")
print(f"  Accumulated u = {int(proof.instance.u)}")

## Scaling: 10 to 1000 Steps

The key property of IVC: **per-step cost is constant.** Let's verify this empirically.

In [ ]:
step_counts = [10, 50, 100, 200, 500, 1000]
prove_times = []
verify_times = []

for n in step_counts:
    z0 = GF(np.array([1, 1]))
    
    t0 = time.time()
    proof = ivc_prove(shape, fibonacci_step_fn, fibonacci_witness_fn, z0, num_steps=n)
    pt = time.time() - t0
    
    t0 = time.time()
    valid = ivc_verify(shape, proof)
    vt = time.time() - t0
    
    prove_times.append(pt)
    verify_times.append(vt)
    
    per_step = pt / n * 1000
    print(f"  n={n:>5}: prove={pt*1000:>8.1f}ms  verify={vt*1000:>6.3f}ms  "
          f"per_step={per_step:.2f}ms  valid={valid}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Prove time should scale linearly with n
axes[0].plot(step_counts, [t*1000 for t in prove_times], 'o-', color='#2563eb', linewidth=2)
axes[0].set_xlabel('Number of steps (n)', fontsize=12)
axes[0].set_ylabel('Prove time (ms)', fontsize=12)
axes[0].set_title('Prover Time vs Steps', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Per-step cost should be roughly constant
per_step_ms = [t/n*1000 for t, n in zip(prove_times, step_counts)]
axes[1].plot(step_counts, per_step_ms, 'o-', color='#dc2626', linewidth=2)
axes[1].set_xlabel('Number of steps (n)', fontsize=12)
axes[1].set_ylabel('Time per step (ms)', fontsize=12)
axes[1].set_title('Per-Step Cost (should be ~constant)', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ivc_scaling.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ivc_scaling.png")

## Verification is Constant-Time

Regardless of how many steps were folded, the verifier checks **one** Relaxed R1CS instance. The verification cost doesn't depend on $n$ at all.

In [ ]:
print("Verification times:")
for n, vt in zip(step_counts, verify_times):
    print(f"  n={n:>5}: {vt*1000:.3f} ms")
print(f"\nVerification is independent of n — always checking 1 accumulated instance.")

## What the Accumulator Looks Like

After folding, the accumulated instance has:
- $u \neq 1$ (sum of all the $u$ contributions)
- $\mathbf{E} \neq \mathbf{0}$ (accumulated cross-terms from every fold)
- Same dimensions as a single-step instance

In [ ]:
# Run 20 steps and inspect the accumulator at each step
from nano_nova.commitment import commit
from nano_nova.r1cs import r1cs_to_relaxed, trivial_relaxed
from nano_nova.folding import fold

acc_inst, acc_wit = trivial_relaxed(shape, commit)
z = GF(np.array([1, 1]))

u_values = []
e_magnitudes = []

for i in range(20):
    z_next = fibonacci_step_fn(z)
    w = fibonacci_witness_fn(z)
    x = GF(np.concatenate([z, z_next]))
    fresh_inst, fresh_wit = r1cs_to_relaxed(shape, x, w, commit)
    acc_inst, acc_wit = fold(shape, acc_inst, acc_wit, fresh_inst, fresh_wit)
    
    u_values.append(int(acc_inst.u))
    # Magnitude of E (treating large field elements as negative when > p/2)
    p = int(GF.order)
    e_mag = sum(min(int(v), p - int(v)) for v in acc_wit.E)
    e_magnitudes.append(e_mag)
    z = z_next

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(range(1, 21), u_values, 'o-', color='#2563eb')
axes[0].set_xlabel('Fold step')
axes[0].set_ylabel('u value')
axes[0].set_title('Relaxation scalar u over folds')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, 21), e_magnitudes, 'o-', color='#dc2626')
axes[1].set_xlabel('Fold step')
axes[1].set_ylabel('||E|| (sum of min magnitudes)')
axes[1].set_title('Error vector magnitude over folds')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('accumulator_evolution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: accumulator_evolution.png")

## Summary: How Nova's IVC Works

1. **Initialize** with a trivial accumulator ($u=0$, $\mathbf{E}=\mathbf{0}$)
2. **At each step:** execute $f$, create a fresh R1CS instance, fold it into the accumulator
3. **Cost per step:** one commitment + field operations (independent of step number)
4. **Final verification:** check one accumulated Relaxed R1CS instance

In real Nova, the final step would produce a SNARK proof of the accumulated instance. Here we directly check the relaxed relation — but the principle is the same.

### What We've Covered

| Notebook | Topic |
|----------|-------|
| 01 | Finite field arithmetic — the substrate |
| 02 | R1CS — encoding computations as constraints |
| 03 | Relaxed R1CS — why naive folding fails, and the fix |
| 04 | The folding step — cross-term, commit, challenge, fold |
| **05** | **IVC — chaining folds for incrementally verifiable computation** |

### Next Steps

- **Notebook 06** (coming): Norm growth analysis for lattice-based folding (LatticeFold)
- Read the [Nova paper](https://eprint.iacr.org/2021/370) for the full construction
- Explore [HyperNova](https://eprint.iacr.org/2023/573) for the CCS generalization with sumcheck